# FLAG 2027 - Notebook 36 RUN-ALL: new loss terms, cosine scoring, fixed attribute-consistency term (on top of system E)

What has carried over from CV to the dev board so far: changes to the **loss** on the scored embeddings (InfoNCE -4.0, SupCon L_out -0.5). What has not: head width/dropout searches (NB34: -1.1 in CV, +0.4 on dev), more identities, more voice encoders, head averaging. So this round only tests things that change the loss or add new information:

| Name | What | Why |
|---|---|---|
| `M1_intra` | L_out + intra-modal SupCon (face-face, voice-voice) | each branch is identity-clustered on its own, not only across modalities (full SupCon, Khosla 2020) |
| `M2_sgm10` / `M2_sgm20` | L_out with a margin of 0.10 / 0.20 on same-gender impostor pairs | the gender track's negatives are all same-gender; make them the hard ones during training |
| cosine | score every trained head by cosine instead of -L2 | paired comparison on the same heads, no retraining |
| attribute term | score - lg*\|P(female\|face) - P(female\|voice)\| - la*\|age(face) - age(voice)\| | fixed per-trial consistency term from frozen attribute models; explicitly allowed by the organizers' FAQ; weights chosen on CV, fixed before any test file |

`M0_ref` = system E's recipe re-trained here. Stage A = usual folds, seeds 1-3; the top 2 loss variants are re-checked on NEW folds with NEW seeds (stage B); only stage B counts. All inference stays per-trial; no test statistics anywhere.

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. About 3-3.5 h. Resumable (Run all again; finished runs are skipped). Paste the **SUMMARY**.


In [ ]:
# Cell 1 - Drive, GPU (hard check)
from google.colab import drive
drive.mount('/content/drive')
import torch, shutil
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run Cell 1 again.'
dev = 'cuda'
print('[NB36 v1 5-Nov] Device:', torch.cuda.get_device_name(0), '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - features, labels, CV folds (identical to NB24), harness with the new recipe options
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
LISTS = '/content/flag_lists'
if not os.path.exists(LISTS + '/done'):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LISTS, members=[m for m in zf.namelist() if m.endswith(('.txt', '.csv'))])
    open(LISTS + '/done', 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(LISTS) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk']), 'mav_lang.npz does not match the feature files - tell Claude'
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in FEAT:
    for s in ['train', 'mav', 'extra']:
        assert s in FEAT[name] and len(FEAT[name][s]) == {'train': len(y), 'mav': len(MAV['spk']), 'extra': len(EXTRA['spk'])}[s], '%s/%s missing or wrong length - tell Claude' % (name, s)
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t(), ft, vt
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def nce_loss(ft, vt, lab, tau):
    """symmetric identity-aware InfoNCE between the two scored embeddings: every same-identity row counts as a positive"""
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; same = lab[:, None] == lab[None, :]
    def one(S_):
        logp = S_ - torch.logsumexp(S_, 1, keepdim=True); return -torch.logsumexp(logp.masked_fill(~same, -1e4), 1).mean()
    return 0.5 * (one(S) + one(S.t()))
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_head(F_raw, V_raw, all_tr, vsets=None, seed=1, cfg=None, max_epochs=MAX_EP):
    """cfg keys: align in {None,'mse','nce'}, w (alignment weight), tau, repair (bool), p_in (input dropout)"""
    cfg = cfg or {}; align, w_al, tau, repair, p_in = cfg.get('align'), cfg.get('w', 1.0), cfg.get('tau', 0.07), cfg.get('repair', False), cfg.get('p_in', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            jv_np = np.concatenate([g[rng.permutation(K_CLIP)] for g in j_np.reshape(P_ID, K_CLIP)]) if repair else j_np   # voice from another clip of the same identity
            j, jv = torch.tensor(j_np, device=dev), torch.tensor(jv_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[jv]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if align == 'mse':
                loss = loss + w_al * ((ft - vt) ** 2).mean()
            elif align == 'nce':
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; VOICE = R4 + ['whisper']
F_RAW = build_train(['clip']); V_full = build_train(VOICE); U_NAP, n_d = nap_basis(V_full[N_CH:], NAP_K); V_RAW = nap_apply(V_full, U_NAP)
note('harness ready: %d challenge rows, %d MAV rows, face %d-d, voice %d-d after NAP-%d (%d language-difference vectors)' % (N_CH, len(EXT), F_RAW.shape[1], V_RAW.shape[1], NAP_K, n_d))

In [ ]:
# NB36 - round 7 on top of system E's loss (SupCon L_out). Needs the two setup cells above.
# Four things that add NEW information or change the LOSS (the only two kinds of change that have carried over to dev so far):
#   M1_intra  : L_out + intra-modal SupCon terms (face-face, voice-voice) so each branch is identity-clustered on its own
#   M2_sgm10  : L_out with a margin of 0.10 on SAME-GENDER different-identity pairs (they are the hard negatives of the gender track)
#   M2_sgm20  : the same with margin 0.20
#   cosine    : scoring every trained head by cosine instead of -L2 (no retraining; paired comparison)
#   attribute : a FIXED consistency term added to the score: -lg*|P(female|face)-P(female|voice)| - la*|age(face)-age(voice)|
#               (face gender probe + face age regressor fitted on the TRAINING fold only; voice attributes from the frozen audeering model;
#                weights chosen on CV, fixed before any test file is touched; per-trial; explicitly allowed by the organizers' FAQ)
# M0_ref = system E's recipe re-trained here (so cosine and attribute columns exist for it). Stage B = NEW folds + NEW seeds; only stage B counts.
SUMMARY = []
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
def lout_loss(ft, vt, lab, tau, margin=None):          # = NB33 c_lout = NB20 lout_loss, plus an optional additive margin matrix (logit units)
    S = Fn.normalize(ft, dim=1) @ Fn.normalize(vt, dim=1).t() / tau; P = (lab[:, None] == lab[None, :]).float()
    if margin is not None:
        S = S + margin
    def one(M):
        return -(P * Fn.log_softmax(M, 1)).sum(1).div(P.sum(1)).mean()
    return 0.5 * (one(S) + one(S.t()))
def intra_loss(z, lab, tau):                          # SupCon L_out inside one modality, self excluded
    Z = Fn.normalize(z, dim=1); S = Z @ Z.t() / tau; eye = torch.eye(len(lab), device=z.device, dtype=torch.bool)
    P = (lab[:, None] == lab[None, :]).float().masked_fill(eye, 0.0); S = S.masked_fill(eye, -1e4)
    return -(P * Fn.log_softmax(S, 1)).sum(1).div(P.sum(1).clamp_min(1.0)).mean()
class Branch2(nn.Module):
    def __init__(self, d_in, d, p):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(p))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP2(FOP):
    def __init__(self, n_class, d_face, d_voice, d=128, p=0.5):
        super().__init__(n_class, d_face, d_voice, d)
        self.face, self.voice = Branch2(d_face, d, p), Branch2(d_voice, d, p)
        for m in list(self.face.modules()) + list(self.voice.modules()):
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
# gender of EVERY training row (challenge rows: official labels; MAV rows: CLIP face probe fitted on the challenge rows, probe acc was 1.00 in NB04)
G_CODE = {g: i for i, g in enumerate(sorted(set(id_gender.values())))}
y_g = np.array([G_CODE[id_gender[c]] for c in y])
probe_all = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1)).fit(F_RAW[:N_CH], y_g)
ROW_G = probe_all.predict(F_RAW); ROW_G[:N_CH] = y_g; ROW_G_T = torch.tensor(ROW_G, device=dev)
note('row gender for the margin: challenge rows from labels, %d MAV rows from the CLIP probe (%.0f%% predicted %s)' % (len(EXT), 100 * ROW_G[N_CH:].mean(), [g for g, i in G_CODE.items() if i == 1][0]))
def train_head6(F_raw, V_raw, all_tr, vsets, seed, cfg):
    w_al, tau, p_in = cfg.get('w', 0.5), cfg.get('tau', 0.07), cfg.get('p_in', 0.0)
    d, p_br, alpha, lr, pid = cfg.get('d', 128), cfg.get('p_br', 0.5), cfg.get('alpha', ALPHA), cfg.get('lr', LR), cfg.get('pid', P_ID)
    w_in, mg = cfg.get('intra', 0.0), cfg.get('margin', 0.0)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}; pid = min(pid, len(cl))
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); gtr = ROW_G_T[tr]; model = FOP2(len(cl), XF.shape[1], XV.shape[1], d, p_br).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    for ep in range(1, MAX_EP + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), pid, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[j]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            margin = None
            if mg > 0:
                g = gtr[j]; sg_neg = ((g[:, None] == g[None, :]) & (lab[:, None] != lab[None, :])).float()
                margin = (mg / tau) * sg_neg                                   # same-gender impostor pairs must be pushed further apart
            loss = Fn.cross_entropy(logits, lab) + alpha * opl_loss(fused, lab) + w_al * lout_loss(ft, vt, lab, tau, margin)
            if w_in > 0:
                loss = loss + w_in * 0.5 * (intra_loss(ft, lab, tau) + intra_loss(vt, lab, tau))
            opt.zero_grad(); loss.backward(); opt.step()
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
@torch.no_grad()
def cos_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return (Fn.normalize(ft, dim=1) * Fn.normalize(vt, dim=1)).sum(1).cpu().numpy()
# ---- attribute-consistency term (post-hoc, fixed weights) ----
AG = None; AGV = None
try:
    da = np.load(os.path.join(FEATDIR, 'audeer.npz')); AG = da['train_ag'].astype(np.float32)            # [age/100, child, female, male] per challenge row
    assert len(AG) == N_CH
except Exception as e:
    note('audeer train_ag not found (%s) -> attribute terms skipped' % repr(e)[:80])
try:
    AGV = np.load(os.path.join(FEATDIR, 'agv.npz'))['train'].astype(np.float32); assert len(AGV) == N_CH   # face age-gender ViT CLS (NB12)
except Exception as e:
    note('agv.npz train not found (%s) -> age term skipped' % repr(e)[:80]); AGV = None
LG = [0.0, 1.0, 2.0, 4.0, 8.0, 16.0]; LA = [0.0, 2.0, 4.0, 8.0, 16.0, 32.0] if AGV is not None else [0.0]
PV_FEM = None
if AG is not None:
    PV_FEM = AG[:, 2] / (AG[:, 2] + AG[:, 3] + 1e-9)                   # P(female | voice) from the frozen audeering model
    # gender code 1 (the probe's positive class) must mean the same gender as the voice probability: decide by the sign of the correlation on labelled rows
    if np.corrcoef(y_g.astype(float), PV_FEM)[0, 1] < 0:
        PV_FEM = 1.0 - PV_FEM
    note('voice gender vs official label on challenge rows: %.1f%% agreement' % (100 * ((PV_FEM > 0.5).astype(int) == y_g).mean()))
def attr_fit(tr_idx):
    """face gender prob + face age regressor fitted on the TRAINING fold's challenge rows only; applied to every challenge row"""
    if AG is None:
        return None
    pf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.1)).fit(F_RAW[tr_idx], y_g[tr_idx]).predict_proba(F_RAW[:N_CH])[:, 1]
    af, av = None, AG[:, 0]
    if AGV is not None:
        id_age = {c: AG[tr_idx][y[tr_idx] == c, 0].mean() for c in np.unique(y[tr_idx])}; tgt = np.array([id_age[c] for c in y[tr_idx]])
        af = make_pipeline(StandardScaler(), Ridge(alpha=100.0)).fit(AGV[tr_idx], tgt).predict(AGV)
    return pf, PV_FEM, af, av
def attr_rows(scores_sets, vsets, fit):
    out = []
    if fit is None:
        return out
    pf, pv, af, av = fit
    for name, (Vi, Fi, L), s in zip(['std', 'gender'], vsets, scores_sets):
        gm = np.abs(pf[Fi] - pv[Vi]); am = np.abs(af[Fi] - av[Vi]) if af is not None else np.zeros(len(Fi))
        for lg in LG:
            for la in LA:
                out.append(dict(set=name, lg=lg, la=la, eer=eer(L, s - lg * gm - la * am)))
    return out
# ---- configs ----
E_CFG = dict(w=0.5, tau=0.07)
CFGS = {'M0_ref': dict(E_CFG), 'M1_intra': dict(E_CFG, intra=0.25), 'M2_sgm10': dict(E_CFG, margin=0.10), 'M2_sgm20': dict(E_CFG, margin=0.20)}
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV36, CSVA = os.path.join(RESDIR, 'nb36_cv.csv'), os.path.join(RESDIR, 'nb36_attr.csv')
if not os.path.exists(CSV36):
    pd.DataFrame(columns=['stage', 'combo', 'seed', 'fold', 'best_ep', 'std', 'gender', 'std_cos', 'gender_cos']).to_csv(CSV36, index=False)
if not os.path.exists(CSVA):
    pd.DataFrame(columns=['stage', 'combo', 'seed', 'fold', 'set', 'lg', 'la', 'eer']).to_csv(CSVA, index=False)
done36 = pd.read_csv(CSV36); t0 = time.time(); ATTR_CACHE = {}
def run36(stage, names, seeds, fold_data):
    global done36
    for name in names:
        for seed in seeds:
            for k in range(K):
                if ((done36.stage == stage) & (done36.combo == name) & (done36.seed == seed) & (done36.fold == k)).any():
                    continue
                tr_idx, vsets = fold_data[k]
                try:
                    model, best_ep, XF, XV = train_head6(F_RAW, V_RAW, np.concatenate([tr_idx, EXT]), vsets, seed, CFGS[name])
                    S_l2 = [same_score(model, XF[Fi], XV[Vi]) for Vi, Fi, L in vsets]; S_cos = [cos_score(model, XF[Fi], XV[Vi]) for Vi, Fi, L in vsets]
                    e_std, e_gen = [eer(L, s) for (Vi, Fi, L), s in zip(vsets, S_l2)]; c_std, c_gen = [eer(L, s) for (Vi, Fi, L), s in zip(vsets, S_cos)]
                    key = (stage, k)
                    if key not in ATTR_CACHE:
                        ATTR_CACHE[key] = attr_fit(tr_idx)
                    arows = attr_rows(S_l2, vsets, ATTR_CACHE[key])
                except Exception as e:
                    print(name, 'FAILED', repr(e)[:300]); best_ep, e_std, e_gen, c_std, c_gen, arows = -1, 50.0, 50.0, 50.0, 50.0, []
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen, std_cos=c_std, gender_cos=c_gen)])
                row.to_csv(CSV36, mode='a', header=False, index=False); done36 = pd.concat([done36, row])
                if arows:
                    A = pd.DataFrame(arows); A.insert(0, 'fold', k); A.insert(0, 'seed', seed); A.insert(0, 'combo', name); A.insert(0, 'stage', stage)
                    A.to_csv(CSVA, mode='a', header=False, index=False)
            print('%s %-9s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
def tab36(stage):
    R = pd.read_csv(CSV36); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2; R['overall_cos'] = (R['std_cos'] + R['gender_cos']) / 2
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall', 'std_cos', 'gender_cos', 'overall_cos', 'best_ep']].mean().groupby('combo').mean().round(2).sort_values('overall')
def attr_tab(stage, combo):
    A = pd.read_csv(CSVA); A = A[(A.stage == stage) & (A.combo == combo)]
    if len(A) == 0:
        return None
    T = A.groupby(['lg', 'la', 'set'])['eer'].mean().unstack('set'); T['overall'] = (T['std'] + T['gender']) / 2
    return T.round(2)
run36('A', list(CFGS), [1, 2, 3], FOLD_DATA)
TA = tab36('A'); refA = TA.loc['M0_ref']; TOP = [c for c in TA.index if c != 'M0_ref'][:2]
run36('B', ['M0_ref'] + TOP, [4, 5, 6], FOLD_B)
TB = tab36('B'); refB = TB.loc['M0_ref']
note('NB36 STAGE A (usual folds, seeds 1-3; system E recipe = M0_ref):'); note(TA.to_string())
note('NB36 STAGE B (NEW folds + NEW seeds 4-6):'); note(TB.to_string())
note('VERDICTS - loss variants (stage B only: overall <= -0.4, or gender <= -0.5 with std <= +0.2 vs M0_ref):')
for c in TOP:
    dB, dg, ds = TB.loc[c, 'overall'] - refB['overall'], TB.loc[c, 'gender'] - refB['gender'], TB.loc[c, 'std'] - refB['std']
    note('%s %s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, json.dumps(CFGS[c]), TA.loc[c, 'overall'] - refA['overall'], dB, ds, dg, 'ADOPT' if (dB <= -0.4 or (dg <= -0.5 and ds <= 0.2)) else 'reject'))
dcos = refB['overall_cos'] - refB['overall']
note('VERDICT - cosine scoring on M0_ref (paired, same heads): stage A %+.2f | stage B %+.2f (std %+.2f, gender %+.2f) -> %s' % (refA['overall_cos'] - refA['overall'], dcos, refB['std_cos'] - refB['std'], refB['gender_cos'] - refB['gender'], 'ADOPT cosine' if dcos <= -0.2 else 'keep -L2'))
TAa, TBa = attr_tab('A', 'M0_ref'), attr_tab('B', 'M0_ref')
if TAa is not None:
    base = TAa.loc[(0.0, 0.0), 'overall']; best = TAa['overall'].idxmin(); lg_, la_ = best
    note('ATTRIBUTE TERM on M0_ref, stage A grid (rows = lg, la; reference lg=la=0 overall %.2f):' % base); note(TAa.sort_values('overall').head(8).to_string())
    note('best on stage A: lg=%.0f la=%.0f -> overall %.2f (std %.2f, gender %.2f)' % (lg_, la_, TAa.loc[best, 'overall'], TAa.loc[best, 'std'], TAa.loc[best, 'gender']))
    if TBa is not None:
        bB, b0 = TBa.loc[best], TBa.loc[(0.0, 0.0)]; dA = bB['overall'] - b0['overall']
        note('stage B with the SAME weights: overall %+.2f (std %+.2f, gender %+.2f) -> %s' % (dA, bB['std'] - b0['std'], bB['gender'] - b0['gender'], 'ADOPT attribute term (lg=%.0f, la=%.0f)' % (lg_, la_) if dA <= -0.3 else 'reject'))
        note('gender-only term (la=0) on stage B: ' + ', '.join('lg=%.0f %+.2f' % (lg, TBa.loc[(lg, 0.0), 'overall'] - b0['overall']) for lg in LG))
        if AGV is not None:
            note('age-only term (lg=0) on stage B: ' + ', '.join('la=%.0f %+.2f' % (la, TBa.loc[(0.0, la), 'overall'] - b0['overall']) for la in LA))
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)
